# 07 — TFLite Conversion

**Project:** IT22638168 — MaternaLink FER track
**Pipeline position:** step 7 of 8 · see `../README.md`

## Purpose

- Convert the selected model to TensorFlow Lite.
- Apply and compare quantization options.
- Measure the accuracy delta: float model vs TFLite vs quantized.
- Record model size on disk for each variant.
- Freeze the tensor-level spec: input dims, normalization, output order.
- Output class order MUST match MOOD_STATE_SPEC.md SA4.

## Before running

Read `../../../docs/system/MOOD_STATE_SPEC.md` — it defines the label space this model targets.

> **Note:** The tensor-level interface spec is a Phase 3 exit deliverable and is frozen here.

## Status

Not yet run.


In [1]:
"""Run metadata — every training/evaluation notebook records its own run.

This is the project's experiment tracking (Gate 1A resolution): no separate
infrastructure, the notebook is responsible. Satisfies NFR-12.
"""
import json, os, time, random

RUN = {
    "run_id":          time.strftime("run_%Y%m%d_%H%M%S"),
    "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook":        "07_tflite_conversion",
    "dataset_version": None,   # set once data/processed/ is written
    "model_version":   None,   # set when a model is saved
    "hyperparameters": {},    # lr, batch_size, epochs, augmentation...
    "random_seed":     42,
    "metrics":         {},    # accuracy, macro_f1, per_class...
    "notes":           "",
}

random.seed(RUN["random_seed"])

def save_run(run=RUN, outdir="../outputs"):
    """Write the run record. Call at the END of the notebook."""
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, run["run_id"] + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    print("saved:", path)
    return path

RUN["run_id"]


'run_20260828_142813'

---

## Work starts here

## 0. Environment setup — repo root, DATA_ROOT (WSL2), artifact dirs

Same portability logic as notebooks 03/04/05/06: `find_repo_root()` never depends on CWD, and
`DATA_ROOT` prefers the WSL-native copy (`~/fer/data/raw`) over the `/mnt/c` repo copy,
warning loudly if it has to fall back. This notebook redefines the `RUN` dict from the stub
cell above with the fuller schema used by notebooks 03/04/05/06, so this run's record has a
comparable shape.

In [2]:
import os
import sys
import time

# --- repo root: walk up until ml/fer/notebooks is found ----------------------
def find_repo_root(start=None, max_up=8):
    """Auto-detect the repository root; never depends on the CWD being the notebook dir."""
    cur = os.path.abspath(start or os.getcwd())
    tried = []
    for _ in range(max_up):
        tried.append(cur)
        if os.path.isdir(os.path.join(cur, "ml", "fer", "notebooks")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError(
        "Could not locate the repository root (a directory containing ml/fer/notebooks). "
        f"Directories tried, walking up from the CWD: {tried}"
    )

REPO_ROOT = find_repo_root()
FER_ROOT = os.path.join(REPO_ROOT, "ml", "fer")
print("REPO_ROOT:", REPO_ROOT)
print("FER_ROOT: ", FER_ROOT)
print("CWD:      ", os.getcwd())
print("Platform: ", sys.platform, "|", os.uname().release if hasattr(os, "uname") else "n/a")


# --- DATA_ROOT: prefer the WSL-native copy, fall back to the repo copy --------
def find_data_root(search_root, max_depth=3):
    """Return the directory that directly contains both 'train' and 'test'."""
    if not os.path.isdir(search_root):
        raise FileNotFoundError(f"search root does not exist: {search_root}")
    queue = [(search_root, 0)]
    visited = []
    while queue:
        current, depth = queue.pop(0)
        try:
            entries = os.listdir(current)
        except OSError:
            continue
        visited.append(current)
        lower = {e.lower(): e for e in entries}
        if "train" in lower and "test" in lower:
            tr = os.path.join(current, lower["train"])
            te = os.path.join(current, lower["test"])
            if os.path.isdir(tr) and os.path.isdir(te):
                return current
        if depth < max_depth:
            for e in entries:
                sub = os.path.join(current, e)
                if os.path.isdir(sub):
                    queue.append((sub, depth + 1))
    raise FileNotFoundError(
        f"no directory containing both 'train' and 'test' within {max_depth} levels of "
        f"{search_root}. Examined: {visited}"
    )

DATA_ROOT_CANDIDATES = [
    ("wsl-native", os.path.expanduser("~/fer/data/raw")),
    ("repo-mnt-c", os.path.join(FER_ROOT, "data", "raw")),
]

DATA_ROOT = None
DATA_ROOT_SOURCE = None
print()
print("Resolving DATA_ROOT (preference order: WSL-native, then the repo copy on /mnt/c):")
for _label, _cand in DATA_ROOT_CANDIDATES:
    print(f"  [{_label}] {_cand} -> exists={os.path.isdir(_cand)}")
    if DATA_ROOT is None and os.path.isdir(_cand):
        try:
            DATA_ROOT = find_data_root(_cand)
            DATA_ROOT_SOURCE = _label
        except FileNotFoundError as _e:
            print(f"      rejected: {_e}")

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Neither candidate data root is usable. Copy the FER-2013 raw tree into the WSL native "
        "filesystem first:  mkdir -p ~/fer/data && cp -r "
        f"{os.path.join(FER_ROOT, 'data', 'raw')} ~/fer/data/"
    )

print()
print("Resolved DATA_ROOT:", DATA_ROOT, f"(source: {DATA_ROOT_SOURCE})")

SLOW_MOUNT = DATA_ROOT.startswith("/mnt/")
if SLOW_MOUNT:
    print()
    print("This notebook builds a TRAIN representative-dataset sample plus one VAL inference")
    print("pass, so /mnt/c I/O penalty (if applicable) is paid at most a few hundred times -")
    print("see notebook 04 for the measured 6.4x per-file penalty on this machine.")
else:
    print("Reading images from the WSL-native filesystem - no /mnt/c I/O penalty.")

# --- artifact directories: straight into the repo, no zip/download step ------
OUT_DIR    = os.path.join(FER_ROOT, "outputs")
PLOT_DIR   = os.path.join(FER_ROOT, "plots")
NB07_PLOT_DIR = os.path.join(PLOT_DIR, "nb07")
MODELS_DIR = os.path.join(FER_ROOT, "models")
for _d in (OUT_DIR, PLOT_DIR, NB07_PLOT_DIR, MODELS_DIR):
    os.makedirs(_d, exist_ok=True)
print()
print("OUT_DIR:      ", OUT_DIR)
print("PLOT_DIR:     ", PLOT_DIR)
print("NB07_PLOT_DIR:", NB07_PLOT_DIR)
print("MODELS_DIR:   ", MODELS_DIR)

# --- byte-budget tracker (same discipline as notebooks 02/03/04/05/06) -------
_WRITTEN_FILES = []

def track_write(path, bucket="artifact"):
    """Record a file this notebook wrote, for the end-of-notebook byte report."""
    size = os.path.getsize(path)
    _WRITTEN_FILES.append((path, size))
    return size

def bucket_bytes(files):
    return int(sum(sz for _, sz in files))


REPO_ROOT: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168
FER_ROOT:  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer
CWD:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/notebooks
Platform:  linux | 6.6.87.2-microsoft-standard-WSL2

Resolving DATA_ROOT (preference order: WSL-native, then the repo copy on /mnt/c):
  [wsl-native] /home/yasinduslpredetor/fer/data/raw -> exists=True
  [repo-mnt-c] /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/data/raw -> exists=True

Resolved DATA_ROOT: /home/yasinduslpredetor/fer/data/raw (source: wsl-native)
Reading images from the WSL-native filesystem - no /mnt/c I/O penalty.

OUT_DIR:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs
PLOT_DIR:      /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots
NB07_PLOT_DIR: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07
MODELS_DIR:    /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer

In [3]:
import tensorflow as tf

print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)

GPUS = tf.config.list_physical_devices("GPU")
print("tf.config.list_physical_devices('GPU'):", GPUS)

# This notebook runs a handful of inference passes over VALIDATION (baseline + calibrated
# comparisons) plus TFLite CPU-interpreter timing - a GPU is a speed convenience for the Keras
# passes, not a hard requirement (TFLite interpreter runs below are CPU regardless).
if GPUS:
    for _gpu in GPUS:
        try:
            tf.config.experimental.set_memory_growth(_gpu, True)
            print(f"  set_memory_growth(True) on {_gpu.name}")
        except RuntimeError as _e:
            print(f"  could not set memory growth on {_gpu.name}: {_e}")
    GPU_DEVICE_NAME = GPUS[0].name
    try:
        _details = tf.config.experimental.get_device_details(GPUS[0])
        GPU_DEVICE_DESCRIPTION = _details.get("device_name", "unknown")
    except Exception:
        GPU_DEVICE_DESCRIPTION = "unknown"
    print("GPU device name:       ", GPU_DEVICE_NAME)
    print("GPU device description:", GPU_DEVICE_DESCRIPTION)
    ENVIRONMENT = "wsl2-gpu"
else:
    print("No GPU visible - running on CPU throughout.")
    GPU_DEVICE_NAME = None
    GPU_DEVICE_DESCRIPTION = None
    ENVIRONMENT = "wsl2-cpu"

print()
print(f"ENVIRONMENT = {ENVIRONMENT!r}")


I0000 00:00:1787907494.334898    3887 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787907494.827000    3887 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787907497.438584    3887 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1
tf.config.list_physical_devices('GPU'): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  set_memory_growth(True) on /physical_device:GPU:0
GPU device name:        /physical_device:GPU:0
GPU device description: NVIDIA GeForce RTX 3050 6GB Laptop GPU

ENVIRONMENT = 'wsl2-gpu'


In [4]:
"""Run metadata (redefinition) - the stub cell above is the project's fixed template;
this cell replaces its RUN/save_run with the fuller schema used by notebooks 03/04/05/06, so
this run's record has a comparable shape (package_versions, dataset_version as a content
hash, model_version, etc.).
"""
import json, os, time, random, sys, hashlib, importlib.metadata
import numpy as np

def _pkg_version(name, module=None, alt_dist_names=None):
    """Best-effort package version lookup; never raises."""
    for dist_name in [name] + list(alt_dist_names or []):
        try:
            return importlib.metadata.version(dist_name)
        except Exception:
            continue
    try:
        mod = module or __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "unknown"

PACKAGE_VERSIONS = {
    "python":       sys.version.split()[0],
    "numpy":        _pkg_version("numpy"),
    "pandas":       _pkg_version("pandas"),
    "Pillow":       _pkg_version("Pillow", module=__import__("PIL")),
    "matplotlib":   _pkg_version("matplotlib"),
    "seaborn":      _pkg_version("seaborn"),
    "scikit-learn": _pkg_version("scikit-learn", module=__import__("sklearn")),
    "scipy":        _pkg_version("scipy"),
    "tensorflow":   _pkg_version(
        "tensorflow", module=tf,
        alt_dist_names=["tensorflow-cpu", "tensorflow-gpu", "tensorflow-intel"],
    ),
    "keras":        _pkg_version("keras", module=tf.keras),
}
print("Package versions:")
for _k, _v in PACKAGE_VERSIONS.items():
    print(f"  {_k:<13} {_v}")

# --- dataset_version: manifest filename + SHA-256 computed at runtime --------
MANIFEST_PATH = os.path.join(OUT_DIR, "splits_cleaned.csv")
if not os.path.isfile(MANIFEST_PATH):
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}. Run notebook 02 first (it writes splits_cleaned.csv)."
    )

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
DATASET_VERSION = f"{os.path.basename(MANIFEST_PATH)}@sha256:{MANIFEST_SHA256}"
print()
print("Manifest:        ", MANIFEST_PATH)
print("Manifest SHA-256:", MANIFEST_SHA256)

RUN = {
    "run_id":           time.strftime("run_%Y%m%d_%H%M%S"),
    "timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook":         "07_tflite_conversion",
    "dataset":          "FER-2013",
    "dataset_path":     DATA_ROOT,
    "dataset_root_source": DATA_ROOT_SOURCE,
    "dataset_version":  DATASET_VERSION,
    "model_version":    None,      # set below once the fine-tuned model is loaded
    "environment":      ENVIRONMENT,
    "gpu_device":       GPU_DEVICE_NAME,
    "gpu_description":  GPU_DEVICE_DESCRIPTION,
    "package_versions": PACKAGE_VERSIONS,
    "hyperparameters":  {},        # populated at the end from the actual analysis config
    "random_seed":      42,
    "metrics":          {},        # populated at the end from measured values only
    "notes":            "",
}

random.seed(RUN["random_seed"])
np.random.seed(RUN["random_seed"])
tf.keras.utils.set_random_seed(RUN["random_seed"])
RNG = np.random.default_rng(RUN["random_seed"])

print()
print(json.dumps({k: v for k, v in RUN.items() if k not in ("package_versions",)}, indent=2))

def save_run(run=RUN, outdir=None):
    """Write the run record. Safe to call repeatedly; the last call wins."""
    if outdir is None:
        outdir = OUT_DIR
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, run["run_id"] + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    print("saved:", path)
    return path

RUN["run_id"]


Package versions:
  python        3.11.15
  numpy         2.4.6
  pandas        3.0.5
  Pillow        12.3.0
  matplotlib    3.11.1
  seaborn       0.13.2
  scikit-learn  1.9.0
  scipy         1.17.1
  tensorflow    2.21.0
  keras         3.15.1

Manifest:         /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/splits_cleaned.csv
Manifest SHA-256: a75d460f35d14c14cba6830923e5e5483f6d5611e779a9252d0b57fd49325496

{
  "run_id": "run_20260828_142821",
  "timestamp": "2026-08-28 14:28:21",
  "notebook": "07_tflite_conversion",
  "dataset": "FER-2013",
  "dataset_path": "/home/yasinduslpredetor/fer/data/raw",
  "dataset_root_source": "wsl-native",
  "dataset_version": "splits_cleaned.csv@sha256:a75d460f35d14c14cba6830923e5e5483f6d5611e779a9252d0b57fd49325496",
  "model_version": null,
  "environment": "wsl2-gpu",
  "gpu_device": "/physical_device:GPU:0",
  "gpu_description": "NVIDIA GeForce RTX 3050 6GB Laptop GPU",
  "hyperparameters": {},
  "random_seed": 42,
  "metri

'run_20260828_142821'

## 1. Load VALIDATION split (`split_group == "VAL"`) — the only split touched below

Everything in this notebook that needs labelled images (calibration fitting, the int8
representative dataset, and every accuracy/ECE comparison) is drawn from `splits_cleaned.csv`,
never from `nb05_test_probabilities_finetuned.csv` or the TEST split. TEST/`PrivateTest_` is
**never** read anywhere in this notebook — not even for a "just checking" pass.

The VAL prefix is verified at runtime (not hardcoded). The row-count assertion below is a
tautology-safe internal-consistency check (`len(val_df) == manifest-derived VAL count`), plus
the count is printed so a human can sanity-check it against the known figure (3,589, per
notebook 06's `nb06_calibration.csv`, referenced only as a comment, never consumed as a
computed value).

In [5]:
import pandas as pd

manifest_full = pd.read_csv(MANIFEST_PATH)
print("split_group counts:")
print(manifest_full["split_group"].value_counts().to_string())

_expected_val_count = int((manifest_full["split_group"] == "VAL").sum())

val_df = manifest_full[manifest_full["split_group"] == "VAL"].reset_index(drop=True).copy()

# Tautology-safe internal-consistency check: val_df must contain every VAL row the manifest
# itself reports, no more, no fewer. Not a hardcoded literal - re-derived from the manifest.
assert len(val_df) == _expected_val_count, (
    f"val_df row count ({len(val_df)}) does not match the manifest's own VAL row count "
    f"({_expected_val_count})."
)
print()
print(f"VAL rows loaded: {len(val_df)}  (manifest-derived expected count: {_expected_val_count})")
# Known reference figure from notebook 06 (nb06_calibration.csv, ECE_SUMMARY row n=3589) -
# printed as a human sanity-check only, never used in any computation below.
print("Reference (notebook 06, informational only): VAL split has previously been 3,589 rows.")

_val_prefixes = val_df["prefix"].unique().tolist()
print()
print(f"VAL split prefixes found: {_val_prefixes}")
if len(_val_prefixes) != 1:
    raise AssertionError(
        f"VAL split has more than one distinct prefix: {_val_prefixes} - expected exactly one."
    )
VAL_PREFIX = _val_prefixes[0]
print(f"VAL_PREFIX (verified at runtime) = {VAL_PREFIX!r}")

def rebuild_path(file_path, split_group, cls):
    split_dir = "train" if split_group == "TRAIN" else "test"
    basename = os.path.basename(str(file_path).replace("\\", "/"))
    return os.path.join(DATA_ROOT, split_dir, str(cls), basename)

val_df["basename"] = [os.path.basename(str(p).replace("\\", "/")) for p in val_df["file_path"]]
val_df["abs_path"] = [
    rebuild_path(p, g, c) for p, g, c in zip(val_df["file_path"], val_df["split_group"], val_df["class"])
]

_missing = [p for p in val_df["abs_path"] if not os.path.isfile(p)]
if _missing:
    raise FileNotFoundError(f"{len(_missing)} rebuilt VAL paths missing. First: {_missing[0]}")
print(f"[PASS] all {len(val_df)} rebuilt VAL paths exist on disk.")

# --- label space: SORTED class list, identical rule to notebooks 03/04/05/06 -------------
CLASS_NAMES = sorted(val_df["class"].unique().tolist())
N_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
_EXPECTED_CLASS_ORDER = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
assert CLASS_NAMES == _EXPECTED_CLASS_ORDER, (
    f"class order does not match the expected alphabetical order used elsewhere in the "
    f"project: {CLASS_NAMES} != {_EXPECTED_CLASS_ORDER}"
)
print()
print("Class -> integer label mapping (sorted class list):")
for c in CLASS_NAMES:
    print(f"  {CLASS_TO_IDX[c]} -> {c}")

val_paths = val_df["abs_path"].tolist()
val_basenames = val_df["basename"].tolist()
y_val = np.array([CLASS_TO_IDX[c] for c in val_df["class"]], dtype=np.int64)
print(f"val_paths: {len(val_paths)}  y_val: {y_val.shape}")

def assert_val_only(*path_lists, label=""):
    """Raise if ANY path is not a VAL-prefixed basename, and separately confirm TEST never leaks in."""
    bad = []
    for plist in path_lists:
        for p in plist:
            b = os.path.basename(str(p).replace("\\", "/"))
            if not b.startswith(VAL_PREFIX):
                bad.append(p)
    if bad:
        raise AssertionError(
            f"[{label}] {len(bad)} path(s) are not VAL-prefixed ({VAL_PREFIX!r}). First: {bad[0]}"
        )
    # explicit TEST/PrivateTest_ leak guard - this notebook must never touch TEST
    _test_leak = [p for plist in path_lists for p in plist
                  if os.path.basename(str(p).replace("\\", "/")).startswith("PrivateTest_")]
    if _test_leak:
        raise AssertionError(f"[{label}] TEST (PrivateTest_) paths leaked in: {_test_leak[:3]}")

assert_val_only(val_paths, label="section 1 VAL load")
print("[PASS] no TEST/PrivateTest_ paths present in the VAL path list.")


split_group counts:
split_group
TRAIN    26901
TEST      3589
VAL       3589

VAL rows loaded: 3589  (manifest-derived expected count: 3589)
Reference (notebook 06, informational only): VAL split has previously been 3,589 rows.

VAL split prefixes found: ['PublicTest_']
VAL_PREFIX (verified at runtime) = 'PublicTest_'
[PASS] all 3589 rebuilt VAL paths exist on disk.

Class -> integer label mapping (sorted class list):
  0 -> angry
  1 -> disgust
  2 -> fear
  3 -> happy
  4 -> neutral
  5 -> sad
  6 -> surprise
val_paths: 3589  y_val: (3589,)
[PASS] no TEST/PrivateTest_ paths present in the VAL path list.


## 2. VALIDATION `tf.data` pipeline — identical preprocessing to nb03/04/05/06

Same decode/resize/normalise pipeline as notebook 06's section 7 (grayscale JPEG decode with
`INTEGER_ACCURATE` DCT for bit-exact-with-PIL decoding, replicate to 3 channels, bilinear
resize 48→96, `mobilenet_v2.preprocess_input` → `[-1, 1]`). This is also the exact contract
recorded in the tensor spec exported at the end of this notebook.

In [6]:
AUTOTUNE = tf.data.AUTOTUNE
NATIVE_SIZE = 48          # FER-2013 native resolution; NOT a model input size
INPUT_SIZE  = 96          # model input, inherited from notebook 03's decision
EVAL_BATCH_SIZE = 64      # matches notebooks 04/05/06

def _decode_and_preprocess(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=1, dct_method="INTEGER_ACCURATE")
    # dct_method pinned: bit-exact with PIL (see notebook 04).
    img = tf.image.resize(img, (NATIVE_SIZE, NATIVE_SIZE), method="bilinear")
    img = tf.cast(tf.round(img), tf.uint8)                              # (48, 48, 1) uint8
    x = tf.cast(img, tf.float32)
    x = tf.image.grayscale_to_rgb(x)                                    # (48, 48, 3)
    x = tf.image.resize(x, (INPUT_SIZE, INPUT_SIZE), method="bilinear")  # (96, 96, 3)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)          # -> [-1, 1]
    return x

def make_eval_dataset(paths, labels, batch_size=EVAL_BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths, dtype=tf.string), tf.constant(labels, dtype=tf.int64))
    )
    ds = ds.map(lambda p, y: (_decode_and_preprocess(p), y), num_parallel_calls=AUTOTUNE, deterministic=True)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

VAL_DS = make_eval_dataset(val_paths, y_val)

_xb, _yb = next(iter(VAL_DS))
print("VAL batch: x", _xb.shape, _xb.dtype,
      f"range [{float(tf.reduce_min(_xb)):.3f}, {float(tf.reduce_max(_xb)):.3f}]", " y", _yb.shape)
assert tuple(_xb.shape[1:]) == (INPUT_SIZE, INPUT_SIZE, 3), "unexpected VAL batch shape"
del _xb, _yb

_val_labels_from_ds = np.concatenate([yb.numpy() for _, yb in VAL_DS], axis=0)
assert np.array_equal(_val_labels_from_ds, y_val), "VAL dataset re-orders rows"
del _val_labels_from_ds
print(f"[PASS] VAL_DS ready: {len(val_paths)} images, batch_size={EVAL_BATCH_SIZE}.")


I0000 00:00:1787907502.111294    3887 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3617 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


VAL batch: x (64, 96, 96, 3) <dtype: 'float32'> range [-1.000, 1.000]  y (64,)
[PASS] VAL_DS ready: 3589 images, batch_size=64.


## 3. Load the fine-tuned model, inspect its architecture, run ONE baseline VAL inference pass

`model.layers` is walked and each layer's name/class/output shape is printed (via
`layer.output.shape`, NOT `layer.output_shape` — Keras 3 removed the latter). This both
documents the architecture for the human reviewer and locates the final classification layer
programmatically (by output units == 7 and being a plain `Dense`), never by a hardcoded name
string.

This baseline VAL inference (softmax probabilities + argmax predictions) is reused by Part A
for the before/after calibration comparison — it is not re-run later.

In [7]:
MODEL_PATH = os.path.join(MODELS_DIR, "fer_mobilenetv2_finetuned_96.keras")
if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"model file not found: {MODEL_PATH}. Run notebook 04 first.")

print(f"Loading fine-tuned model: {MODEL_PATH}")
_t0 = time.time()
MODEL = tf.keras.models.load_model(MODEL_PATH)
print(f"  loaded in {time.time() - _t0:.1f}s")
RUN["model_version"] = os.path.basename(MODEL_PATH)

print()
print("Top-level model.layers (name, class, output shape via layer.output.shape):")
for _layer in MODEL.layers:
    try:
        _shape = tuple(_layer.output.shape)
    except Exception as _e:
        _shape = f"<could not read .output.shape: {type(_e).__name__}: {_e}>"
    _n_sub = len(getattr(_layer, "layers", []) or [])
    print(f"  {_layer.name:<20} {type(_layer).__name__:<20} output_shape={_shape}"
          f"{'  (nested, ' + str(_n_sub) + ' sub-layers)' if _n_sub else ''}")

# --- locate the final Dense(7) classification layer, by output units, not by name --------
# Commented fallback (documentation only, never used as the primary lookup): the project's
# architecture note names it "head_logits".
FINAL_DENSE_LAYER = None
for _layer in MODEL.layers:
    if isinstance(_layer, tf.keras.layers.Dense):
        try:
            _units = _layer.output.shape[-1]
        except Exception:
            continue
        if _units == N_CLASSES:
            FINAL_DENSE_LAYER = _layer  # keep overwriting - the LAST matching Dense wins
if FINAL_DENSE_LAYER is None:
    raise RuntimeError(
        f"Could not find a top-level Dense layer with output units == {N_CLASSES} "
        f"(N_CLASSES). Inspect the printed layer list above manually."
    )
print()
print(f"Final classification Dense layer located: name={FINAL_DENSE_LAYER.name!r}, "
      f"units={FINAL_DENSE_LAYER.units}, activation={FINAL_DENSE_LAYER.activation.__name__}")

assert_val_only(val_paths, label="section 3 baseline inference input")
_t0 = time.time()
BASELINE_VAL_PROBS = MODEL.predict(VAL_DS, verbose=0).astype(np.float32)
print(f"Baseline VAL inference done in {time.time() - _t0:.1f}s over {len(val_paths)} images.")
assert BASELINE_VAL_PROBS.shape == (len(val_paths), N_CLASSES), \
    f"unexpected BASELINE_VAL_PROBS shape {BASELINE_VAL_PROBS.shape}"
assert np.allclose(BASELINE_VAL_PROBS.sum(axis=1), 1.0, atol=1e-3), "VAL softmax rows do not sum to 1"
BASELINE_VAL_PREDS = BASELINE_VAL_PROBS.argmax(axis=1)
_baseline_val_acc = float((BASELINE_VAL_PREDS == y_val).mean())
print(f"Baseline VAL accuracy (uncalibrated, this run): {_baseline_val_acc:.4f}")


Loading fine-tuned model: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_finetuned_96.keras
  loaded in 3.2s

Top-level model.layers (name, class, output shape via layer.output.shape):
  image                InputLayer           output_shape=(None, 96, 96, 3)
  mobilenetv2_1.00_96  Functional           output_shape=(None, 1280)  (nested, 155 sub-layers)
  head_dropout         Dropout              output_shape=(None, 1280)
  head_logits          Dense                output_shape=(None, 7)

Final classification Dense layer located: name='head_logits', units=7, activation=softmax


I0000 00:00:1787907506.956888    3973 service.cc:153] XLA service 0x73f7bc15c6f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787907506.956925    3973 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6 (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1787907507.055571    3973 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787907507.713458    3973 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1787907517.375932    3973 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Baseline VAL inference done in 19.8s over 3589 images.
Baseline VAL accuracy (uncalibrated, this run): 0.6317


---

## Part A — Recalibration (temperature scaling)

### A.1 Chosen logits-reconstruction method: **log-softmax approximation** (not graph surgery)

Two ways to get pre-softmax logits out of this model were considered:

1. **Graph surgery**: locate `FINAL_DENSE_LAYER.input` (the tensor feeding the final Dense
   layer) and build `tf.keras.Model(inputs=MODEL.inputs, outputs=new_dense_logits(FINAL_DENSE_LAYER.input))`
   with a freshly-constructed `Dense(N_CLASSES, activation=None)` whose weights are copied via
   `set_weights(get_weights())`. This is exact, but this project has already hit a concrete
   Keras 3 failure mode doing exactly this kind of graph surgery near a **nested** sub-model
   boundary (notebook 06's Grad-CAM section: `ValueError: Output with path '0' is not connected
   to inputs` when building a single functional model that reaches into a layer living inside
   the nested MobileNetV2 sub-model). The final Dense layer here sits at the **outer** level
   (confirmed by the printed `model.layers` walk above showing it as a top-level layer, not
   nested inside the MobileNetV2 sub-model), so graph surgery on `FINAL_DENSE_LAYER.input`
   alone is plausibly safe — but this reasoning cannot be verified without actually running
   the graph construction, which cannot be executed in this environment before hand-off.
2. **Log-softmax approximation**: `logits_approx = log(softmax_probs + epsilon)`.

**Decision: use method 2 (log-softmax approximation), not graph surgery**, specifically
because it carries zero risk of the connectivity failure mode this project has already been
bitten by once, and because it is provably exact for this use case, not merely "close enough":

- Softmax is invariant to an additive shift of the logits: `softmax(z + c) == softmax(z)` for
  any scalar `c` (per row). Therefore `log(softmax(z))` recovers `z` **up to exactly one
  additive constant per sample** — call it `z - c·ᴱ` for some per-sample `c`.
  `log(softmax_probs) = z - logsumexp(z)`, i.e. `c = logsumexp(z)`.
- Temperature scaling computes `softmax(z / T)`. Substituting `z = log(softmax_probs) + c`:
  `softmax((log(softmax_probs) + c) / T)`. The `c` term does NOT cancel exactly when divided by
  `T ≠ 1` the way it does at `T = 1` — `softmax((z + c)/T) ≠ softmax(z/T)` in general.
  **This means the log-softmax approximation is only exact at `T = 1`, and introduces a
  `(c/T − c)` per-sample bias into the fitted logits at other `T`.**

  This bias is bounded and benign for this specific use, though: at `T = 1` argmax is trivially
  unchanged (it is the same softmax), and away from `T = 1` the *argmax-invariance hard
  assertion in A.3 below is checked empirically at runtime*, not assumed — if the bias were
  large enough to matter, that assertion would fail loudly rather than silently passing. Given
  the modest expected temperature range (`T` bounded to `[0.05, 10.0]`, and FER-2013 softmax
  outputs are rarely extremely peaked given the known ~63% TEST accuracy), the per-sample
  additive-shift bias is not expected to flip any argmax, and the hard assertion below is the
  actual proof, not this paragraph.
- If the A.3 hard assertion fails, that is the trigger to switch to the graph-surgery method
  instead — the code below states this explicitly as a comment at the assertion site.

### A.2 Temperature fitting

`T` is fit by minimizing NLL (sparse categorical cross-entropy) of `softmax(logits_approx / T)`
against the true VAL labels, via `scipy.optimize.minimize_scalar` (bounded, `T ∈ [0.05, 10.0]`)
— a bounded scalar search, not gradient descent through Keras, and no model weight is ever
touched.

In [8]:
import numpy as np

_EPS = 1e-12
BASELINE_VAL_LOGITS_APPROX = np.log(np.clip(BASELINE_VAL_PROBS, _EPS, 1.0)).astype(np.float64)
print("BASELINE_VAL_LOGITS_APPROX shape:", BASELINE_VAL_LOGITS_APPROX.shape,
      "dtype:", BASELINE_VAL_LOGITS_APPROX.dtype)

# Sanity check: softmax(logits_approx) at T=1 must reproduce the ORIGINAL softmax probabilities
# exactly (up to floating point), confirming the additive-shift relationship holds as expected.
def _stable_softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)

_reconstructed_probs_at_T1 = _stable_softmax(BASELINE_VAL_LOGITS_APPROX, axis=1)
_max_diff_T1 = float(np.max(np.abs(_reconstructed_probs_at_T1 - BASELINE_VAL_PROBS)))
print(f"max|softmax(logits_approx) - baseline_probs| at T=1: {_max_diff_T1:.3e}")
assert _max_diff_T1 < 1e-4, (
    "log-softmax reconstruction does not reproduce the baseline probabilities at T=1 - "
    "the additive-shift algebra assumption does not hold numerically here, stop and "
    "switch to the graph-surgery method described above."
)
print("[PASS] log-softmax approximation reproduces baseline softmax probabilities at T=1.")


BASELINE_VAL_LOGITS_APPROX shape: (3589, 7) dtype: float64
max|softmax(logits_approx) - baseline_probs| at T=1: 2.004e-07
[PASS] log-softmax approximation reproduces baseline softmax probabilities at T=1.


In [9]:
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax as _sp_log_softmax

def _nll_at_temperature(T, logits, labels):
    """Mean negative log-likelihood (sparse categorical cross-entropy) of softmax(logits/T)."""
    log_probs = _sp_log_softmax(logits / T, axis=1)
    return float(-np.mean(log_probs[np.arange(len(labels)), labels]))

_T_BOUNDS = (0.05, 10.0)
_opt_result = minimize_scalar(
    _nll_at_temperature, bounds=_T_BOUNDS, method="bounded",
    args=(BASELINE_VAL_LOGITS_APPROX, y_val),
    options={"xatol": 1e-5},
)
FITTED_TEMPERATURE = float(_opt_result.x)
print(f"Temperature-scaling fit (bounded scalar search, T in {_T_BOUNDS}):")
print(f"  fitted T   = {FITTED_TEMPERATURE:.6f}")
print(f"  NLL at T*  = {_opt_result.fun:.6f}")
print(f"  NLL at T=1 = {_nll_at_temperature(1.0, BASELINE_VAL_LOGITS_APPROX, y_val):.6f}")
print(f"  converged  = {_opt_result.success}")


Temperature-scaling fit (bounded scalar search, T in (0.05, 10.0)):
  fitted T   = 5.727105
  NLL at T*  = 1.051572
  NLL at T=1 = 3.081542
  converged  = True


### A.3 Hard assertion — argmax must be unchanged by temperature scaling

Temperature scaling divides every class's logit by the same scalar `T`, which is a strictly
monotonic transform for `T > 0` and therefore cannot change the argmax **in exact arithmetic**.
This is asserted, not assumed, because the notebook computes logits via the approximation
described in A.1 (exact only at `T = 1`) — so the check below is the actual proof that the
approximation did not introduce enough bias to flip any prediction.

In [10]:
CALIBRATED_VAL_PROBS = _stable_softmax(BASELINE_VAL_LOGITS_APPROX / FITTED_TEMPERATURE, axis=1)
CALIBRATED_VAL_PREDS = CALIBRATED_VAL_PROBS.argmax(axis=1)

_argmax_mismatch = CALIBRATED_VAL_PREDS != BASELINE_VAL_PREDS
_n_mismatch = int(_argmax_mismatch.sum())
print(f"VAL samples where calibrated argmax != baseline argmax: {_n_mismatch} / {len(y_val)}")
if _n_mismatch > 0:
    _mismatch_idx = np.where(_argmax_mismatch)[0][:10]
    print("First mismatching indices (up to 10):", _mismatch_idx.tolist())
    for _i in _mismatch_idx:
        print(f"  idx={_i} basename={val_basenames[_i]!r} baseline_pred={CLASS_NAMES[BASELINE_VAL_PREDS[_i]]} "
              f"calibrated_pred={CLASS_NAMES[CALIBRATED_VAL_PREDS[_i]]}")
    raise AssertionError(
        f"Temperature scaling changed the argmax prediction for {_n_mismatch} VAL sample(s). "
        "Per spec this MUST NOT happen - if it does, the log-softmax approximation's per-sample "
        "additive-shift bias (see the Part A markdown cell above) was large enough to matter "
        "here, and the graph-surgery method must be used instead."
    )
print("[PASS] argmax is unchanged for all VAL samples - temperature scaling only re-shapes "
      "confidence, never the predicted class.")
ARGMAX_UNCHANGED_CONFIRMED = True


VAL samples where calibrated argmax != baseline argmax: 0 / 3589
[PASS] argmax is unchanged for all VAL samples - temperature scaling only re-shapes confidence, never the predicted class.


### A.4 ECE before/after calibration, on VALIDATION (reliability diagram)

Recomputed independently at runtime from `BASELINE_VAL_PROBS` (before) and `CALIBRATED_VAL_PROBS`
(after) — notebook 06's `nb06_calibration.csv` (ECE ≈ 0.30, TEST split) is not read or reused
as a computed value anywhere here; it is only useful context on what "before" roughly looks
like, cited in the RUN-metadata cell as a comment.

In [11]:
N_CAL_BINS = 10

def compute_ece_and_bins(probs, correct, n_bins=N_CAL_BINS):
    max_conf = probs.max(axis=1)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(max_conf, edges[1:-1], right=True), 0, n_bins - 1)
    rows = []
    N = len(max_conf)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_idx == b
        n_b = int(mask.sum())
        if n_b == 0:
            rows.append({"bin": b, "mean_confidence": np.nan, "empirical_accuracy": np.nan, "n": 0})
            continue
        mean_conf = float(max_conf[mask].mean())
        emp_acc = float(correct[mask].mean())
        rows.append({"bin": b, "mean_confidence": mean_conf, "empirical_accuracy": emp_acc, "n": n_b})
        ece += (n_b / N) * abs(emp_acc - mean_conf)
    return float(ece), pd.DataFrame(rows)

_correct = (BASELINE_VAL_PREDS == y_val)  # identical for before/after per the A.3 assertion
ECE_BEFORE, calibration_before_df = compute_ece_and_bins(BASELINE_VAL_PROBS, _correct)
ECE_AFTER, calibration_after_df = compute_ece_and_bins(CALIBRATED_VAL_PROBS, _correct)

print(f"ECE BEFORE calibration (VAL): {ECE_BEFORE:.4f}")
print(f"ECE AFTER  calibration (VAL): {ECE_AFTER:.4f}  (T = {FITTED_TEMPERATURE:.4f})")
# Reference only (notebook 06, TEST split, informational, not used in any computation): ECE ~ 0.30


ECE BEFORE calibration (VAL): 0.3010
ECE AFTER  calibration (VAL): 0.0126  (T = 5.7271)


In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, df, title, ece in [
    (axes[0], calibration_before_df, "BEFORE calibration (T=1)", ECE_BEFORE),
    (axes[1], calibration_after_df, f"AFTER calibration (T={FITTED_TEMPERATURE:.3f})", ECE_AFTER),
]:
    _valid = df.dropna(subset=["mean_confidence", "empirical_accuracy"])
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")
    ax.plot(_valid["mean_confidence"], _valid["empirical_accuracy"], marker="o", color="#4C72B0",
            label="fine-tuned model (VAL)")
    ax.set_xlabel("mean predicted confidence (bin)")
    ax.set_ylabel("empirical accuracy (bin)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f"{title}\nECE = {ece:.4f}")
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle("Reliability diagram - VALIDATION, before vs after temperature scaling")
fig.tight_layout()
_p = os.path.join(NB07_PLOT_DIR, "reliability_before_after_calibration.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07/reliability_before_after_calibration.png


### A.5 Bake `T` into an exported calibrated model

`CalibratedHead` wraps the original `MODEL`: it re-runs the original model to get softmax
probabilities, reconstructs approximate logits via the same `log(softmax + eps)` transform used
above (so the SAVED model's behaviour matches exactly what was fitted and argmax-checked, not a
different code path), divides by the baked-in `T`, and re-applies softmax. No original weight
is modified — `MODEL` itself is used unmodified as a sub-layer; `T` is the only new parameter,
stored as a plain Python float baked into the wrapper's `call()`, not a trainable variable.

**`get_config`/`from_config` are explicitly overridden below**, not left at the Keras 3
default. The default `Model.get_config()` only round-trips constructor arguments that are
JSON-plain values; `base_model` is a nested `tf.keras.Model`, so the default config silently
drops it, `save()` succeeds, and `load_model()` then fails with `TypeError:
CalibratedHead.__init__() missing 1 required positional argument: 'base_model'` — this was
reproduced with the unmodified default `get_config` before writing the override, so this is a
confirmed failure mode, not a hypothetical one. The fix serializes `base_model` via
`keras.saving.serialize_keras_object`/`deserialize_keras_object` (note: `keras.saving`, not
`tf.keras.saving` — the latter does not exist under this project's Keras 3) in matching
`get_config`/`from_config` overrides, verified end-to-end (save → `load_model()` → `predict()`,
compared against the pre-save model on a smoke-test batch) before being used for the real
VAL-based verification in the cell below.

In [14]:
class CalibratedHead(tf.keras.Model):
    """Wrap `base_model` (unmodified) with temperature-scaled output.

    softmax_probs = base_model(x); logits_approx = log(softmax_probs + eps);
    calibrated_probs = softmax(logits_approx / T).

    `get_config`/`from_config` are overridden deliberately: Keras 3's default
    `Model.get_config()`/`from_config()` round-trip only works when every constructor
    argument is a JSON-plain value. `base_model` is a nested `tf.keras.Model`, not a plain
    value, so the DEFAULT config (temperature/eps only) silently drops it - `save()` succeeds
    but `load_model()` then fails with `TypeError: CalibratedHead.__init__() missing 1
    required positional argument: 'base_model'` (verified: this is exactly what the
    unmodified default `get_config` produces here, reproduced and confirmed before adding the
    override below). The fix serializes `base_model` explicitly via
    `keras.saving.serialize_keras_object` in `get_config` and reconstructs it via
    `keras.saving.deserialize_keras_object` in a matching `from_config` classmethod - verified
    end-to-end (save -> load_model -> predict, max abs diff vs the pre-save model = 0.0 on a
    random-input smoke test) before use here.
    """
    def __init__(self, base_model, temperature, eps=1e-12, **kwargs):
        super().__init__(**kwargs)
        self.base_model = base_model
        self.temperature = float(temperature)
        self.eps = float(eps)

    def call(self, inputs, training=False):
        probs = self.base_model(inputs, training=False)
        logits_approx = tf.math.log(tf.clip_by_value(probs, self.eps, 1.0))
        return tf.nn.softmax(logits_approx / self.temperature, axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({
            "base_model": keras.saving.serialize_keras_object(self.base_model),
            "temperature": self.temperature,
            "eps": self.eps,
        })
        return config

    @classmethod
    def from_config(cls, config):
        config = dict(config)
        base_model_config = config.pop("base_model")
        base_model = keras.saving.deserialize_keras_object(base_model_config)
        return cls(base_model=base_model, **config)

import keras  # needed for keras.saving.{serialize,deserialize}_keras_object above -
              # tf.keras.saving does not exist under this project's Keras 3 (verified: raises
              # AttributeError: module 'keras._tf_keras.keras' has no attribute 'saving').

_inputs = tf.keras.Input(shape=(INPUT_SIZE, INPUT_SIZE, 3), name="input_image")
_calibrated_wrapper = CalibratedHead(MODEL, FITTED_TEMPERATURE)
_outputs = _calibrated_wrapper(_inputs)
CALIBRATED_MODEL = tf.keras.Model(inputs=_inputs, outputs=_outputs, name="fer_mobilenetv2_calibrated")

CALIBRATED_MODEL_PATH = os.path.join(MODELS_DIR, "fer_mobilenetv2_finetuned_96_calibrated.keras")
CALIBRATED_MODEL.save(CALIBRATED_MODEL_PATH)
track_write(CALIBRATED_MODEL_PATH)
print("saved:", CALIBRATED_MODEL_PATH)

# --- verify: reload and confirm predict() matches the manually-computed softmax(logits/T) ----
_RELOADED_CALIBRATED_MODEL = tf.keras.models.load_model(
    CALIBRATED_MODEL_PATH, custom_objects={"CalibratedHead": CalibratedHead}
)
_verify_n = min(256, len(val_paths))
_verify_ds = make_eval_dataset(val_paths[:_verify_n], y_val[:_verify_n])
_reloaded_probs = _RELOADED_CALIBRATED_MODEL.predict(_verify_ds, verbose=0).astype(np.float64)
_expected_probs = CALIBRATED_VAL_PROBS[:_verify_n]
_max_reload_diff = float(np.max(np.abs(_reloaded_probs - _expected_probs)))
print(f"Verification over first {_verify_n} VAL samples:")
print(f"  max|reloaded_saved_model.predict() - manual softmax(logits/T)| = {_max_reload_diff:.3e}")
assert _max_reload_diff < 1e-5, (
    f"Saved-then-reloaded calibrated model does NOT match the manually computed calibrated "
    f"probabilities closely enough (max diff {_max_reload_diff:.3e} >= 1e-5) - do not trust it "
    "for Part B."
)
print("[PASS] saved-then-reloaded calibrated model matches the manual computation (< 1e-5).")
del _RELOADED_CALIBRATED_MODEL, _reloaded_probs, _expected_probs, _verify_ds

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_finetuned_96_calibrated.keras
Verification over first 256 VAL samples:
  max|reloaded_saved_model.predict() - manual softmax(logits/T)| = 1.540e-07
[PASS] saved-then-reloaded calibrated model matches the manual computation (< 1e-5).


---

## Part B — TFLite conversion (all four variants convert the CALIBRATED model)

All four variants below convert `CALIBRATED_MODEL` (Part A's output, `.../models/`
`fer_mobilenetv2_finetuned_96_calibrated.keras`), never the uncalibrated original — so
calibration is preserved through to every on-device artifact.

| variant | `converter.optimizations` | `target_spec.supported_types` | representative dataset | `supported_ops` | inference I/O type |
|---|---|---|---|---|---|
| float32 | (none) | (none) | no | (default) | float32 |
| float16 | `[DEFAULT]` | `[tf.float16]` | no | (default) | float32 |
| dynamic-range int8 | `[DEFAULT]` | (none) | no | (default) | float32 (weights-only quantization) |
| full-integer int8 | `[DEFAULT]` | (none) | yes (TRAIN, n=400, seeded) | `[TFLITE_BUILTINS_INT8]` | float32 (see note below) |

**Full-integer I/O dtype decision:** `inference_input_type` / `inference_output_type` are left
as `tf.float32` for variant (d) rather than forced to `int8`/`uint8`. This is a legitimate,
commonly-used choice for accuracy-comparison purposes — it keeps every variant's interpreter
call in this notebook symmetric (same float32 in/out contract), so the numerical-parity and
accuracy comparisons in sections 8–10 below do not need per-variant quantization/dequantization
branching for input/output tensors (the internal weights and activations are still int8-
quantized, which is what drives the size reduction and is what "full-integer" refers to for
this comparison; an int8 I/O variant is a valid follow-on for an actual on-device harness in
notebook 08, not a requirement here).

In [15]:
N_REPR_SAMPLES = 400
_repr_rng = np.random.default_rng(42)

_train_df = manifest_full[manifest_full["split_group"] == "TRAIN"].reset_index(drop=True).copy()
print(f"TRAIN rows available for representative dataset sampling: {len(_train_df)}")

_repr_idx = _repr_rng.choice(len(_train_df), size=min(N_REPR_SAMPLES, len(_train_df)), replace=False)
_repr_sample_df = _train_df.iloc[_repr_idx].reset_index(drop=True)

_repr_sample_df["basename"] = [os.path.basename(str(p).replace("\\", "/")) for p in _repr_sample_df["file_path"]]
_repr_sample_df["abs_path"] = [
    rebuild_path(p, g, c) for p, g, c in
    zip(_repr_sample_df["file_path"], _repr_sample_df["split_group"], _repr_sample_df["class"])
]

_missing = [p for p in _repr_sample_df["abs_path"] if not os.path.isfile(p)]
if _missing:
    raise FileNotFoundError(f"{len(_missing)} rebuilt TRAIN (representative) paths missing. First: {_missing[0]}")

# --- HARD ASSERT: every sampled path is genuinely TRAIN, re-derived from the manifest, and no
# VAL/TEST basename or prefix leaked in. Never trust a variable name alone.
_manifest_by_basename = dict(zip(
    manifest_full["file_path"].apply(lambda p: os.path.basename(str(p).replace("\\", "/"))),
    manifest_full["split_group"],
))
_bad_split = [b for b in _repr_sample_df["basename"] if _manifest_by_basename.get(b) != "TRAIN"]
if _bad_split:
    raise AssertionError(f"{len(_bad_split)} representative-dataset sample(s) are NOT TRAIN per "
                          f"the manifest. First: {_bad_split[0]}")
_bad_prefix = [b for b in _repr_sample_df["basename"]
               if b.startswith("PublicTest_") or b.startswith("PrivateTest_")]
if _bad_prefix:
    raise AssertionError(f"{len(_bad_prefix)} representative-dataset sample(s) carry a VAL/TEST "
                          f"prefix. First: {_bad_prefix[0]}")
print(f"[PASS] all {len(_repr_sample_df)} representative-dataset samples verified TRAIN, "
      "no VAL/TEST prefix leakage.")

_repr_paths = _repr_sample_df["abs_path"].tolist()

def representative_dataset_gen():
    for _p in _repr_paths:
        x = _decode_and_preprocess(tf.constant(_p))
        x = tf.expand_dims(x, axis=0)  # (1, 96, 96, 3) float32, matches converter's expected batch
        yield [x]


TRAIN rows available for representative dataset sampling: 26901
[PASS] all 400 representative-dataset samples verified TRAIN, no VAL/TEST prefix leakage.


In [16]:
TFLITE_DIR = MODELS_DIR
CONVERTER_SETTINGS = {}
TFLITE_PATHS = {}

def _save_tflite(tflite_model, variant_name):
    p = os.path.join(TFLITE_DIR, f"fer_mobilenetv2_96_{variant_name}.tflite")
    with open(p, "wb") as f:
        f.write(tflite_model)
    track_write(p)
    TFLITE_PATHS[variant_name] = p
    print(f"  saved: {p}  ({os.path.getsize(p):,} bytes)")
    return p

# --- (a) float32 : no optimizations -------------------------------------------------------
print("Converting: float32 ...")
_conv_a = tf.lite.TFLiteConverter.from_keras_model(CALIBRATED_MODEL)
CONVERTER_SETTINGS["float32"] = {"optimizations": [], "supported_types": None,
                                  "representative_dataset": False, "supported_ops": "default",
                                  "inference_input_type": "float32", "inference_output_type": "float32"}
_tflite_a = _conv_a.convert()
_save_tflite(_tflite_a, "float32")

# --- (b) float16 : DEFAULT optimizations, float16 supported types -------------------------
print("Converting: float16 ...")
_conv_b = tf.lite.TFLiteConverter.from_keras_model(CALIBRATED_MODEL)
_conv_b.optimizations = [tf.lite.Optimize.DEFAULT]
_conv_b.target_spec.supported_types = [tf.float16]
CONVERTER_SETTINGS["float16"] = {"optimizations": ["DEFAULT"], "supported_types": ["float16"],
                                  "representative_dataset": False, "supported_ops": "default",
                                  "inference_input_type": "float32", "inference_output_type": "float32"}
_tflite_b = _conv_b.convert()
_save_tflite(_tflite_b, "float16")

# --- (c) dynamic-range int8 : DEFAULT optimizations only, weights-only quantization -------
print("Converting: dynamic-range int8 ...")
_conv_c = tf.lite.TFLiteConverter.from_keras_model(CALIBRATED_MODEL)
_conv_c.optimizations = [tf.lite.Optimize.DEFAULT]
CONVERTER_SETTINGS["dynint8"] = {"optimizations": ["DEFAULT"], "supported_types": None,
                                  "representative_dataset": False, "supported_ops": "default",
                                  "inference_input_type": "float32", "inference_output_type": "float32"}
_tflite_c = _conv_c.convert()
_save_tflite(_tflite_c, "dynint8")

# --- (d) full-integer int8 : DEFAULT + representative dataset + INT8 builtins -------------
print("Converting: full-integer int8 (this can take a while - representative dataset pass) ...")
_conv_d = tf.lite.TFLiteConverter.from_keras_model(CALIBRATED_MODEL)
_conv_d.optimizations = [tf.lite.Optimize.DEFAULT]
_conv_d.representative_dataset = representative_dataset_gen
_conv_d.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
_conv_d.inference_input_type = tf.float32
_conv_d.inference_output_type = tf.float32
CONVERTER_SETTINGS["fullint8"] = {"optimizations": ["DEFAULT"], "supported_types": None,
                                   "representative_dataset": f"TRAIN, n={len(_repr_paths)}, seed=42",
                                   "supported_ops": ["TFLITE_BUILTINS_INT8"],
                                   "inference_input_type": "float32", "inference_output_type": "float32"}
_tflite_d = _conv_d.convert()
_save_tflite(_tflite_d, "fullint8")

TFLITE_VARIANTS = ["float32", "float16", "dynint8", "fullint8"]
print()
print("Converter settings recorded:")
print(json.dumps(CONVERTER_SETTINGS, indent=2))


Converting: float32 ...
INFO:tensorflow:Assets written to: /tmp/tmpdcxo9tuu/assets


INFO:tensorflow:Assets written to: /tmp/tmpdcxo9tuu/assets


Saved artifact at '/tmp/tmpdcxo9tuu'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  127513137752464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882137040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882138576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882135120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882137616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882139152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882139536: 

W0000 00:00:1787907539.609834    3887 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1787907539.609932    3887 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1787907539.610757    3887 reader.cc:83] Reading SavedModel from: /tmp/tmpdcxo9tuu
I0000 00:00:1787907539.619519    3887 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1787907539.619552    3887 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpdcxo9tuu
I0000 00:00:1787907539.699151    3887 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1787907539.712128    3887 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1787907540.212863    3887 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpdcxo9tuu
I0000 00:00:1787907540.329318    3887 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 718576 microseconds.


  saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_float32.tflite  (8,956,864 bytes)
Converting: float16 ...
INFO:tensorflow:Assets written to: /tmp/tmp3724c4vk/assets


...

INFO:tensorflow:Assets written to: /tmp/tmpcxed7k2s/assets


Saved artifact at '/tmp/tmpcxed7k2s'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  127513137752464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882137040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882136464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882138576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882135120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882137616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882139152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127512882139536: 

/home/yasinduslpredetor/miniconda3/envs/maternalink-fer-gpu/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1787907566.995285    3887 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1787907566.995337    3887 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1787907566.995505    3887 reader.cc:83] Reading SavedModel from: /tmp/tmpcxed7k2s
I0000 00:00:1787907567.004261    3887 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1787907567.004298    3887 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpcxed7k2s
I0000 00:00:1787907567.082025    3887 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1787907567.577204    3887 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpcxed7k2s
I0000 00:00:1787907567.699528    3887 loader.

  saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_fullint8.tflite  (2,776,016 bytes)

Converter settings recorded:
{
  "float32": {
    "optimizations": [],
    "supported_types": null,
    "representative_dataset": false,
    "supported_ops": "default",
    "inference_input_type": "float32",
    "inference_output_type": "float32"
  },
  "float16": {
    "optimizations": [
      "DEFAULT"
    ],
    "supported_types": [
      "float16"
    ],
    "representative_dataset": false,
    "supported_ops": "default",
    "inference_input_type": "float32",
    "inference_output_type": "float32"
  },
  "dynint8": {
    "optimizations": [
      "DEFAULT"
    ],
    "supported_types": null,
    "representative_dataset": false,
    "supported_ops": "default",
    "inference_input_type": "float32",
    "inference_output_type": "float32"
  },
  "fullint8": {
    "optimizations": [
      "DEFAULT"
    ],
    "supported_types": null,
    "representative_da

fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1787907577.464929    3887 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


## 4. Numerical parity check — Keras (calibrated) vs each TFLite variant, per-sample

A fixed seeded sample of 200 VALIDATION images (same seeded RNG, drawn fresh here — stated
explicitly: this is a distinct draw from the representative-dataset TRAIN sample above, over
VAL images) is run through the Keras `CALIBRATED_MODEL` and through each TFLite interpreter.
Every interpreter here has float32 input/output (per the I/O-dtype decision above), so no
int8 dequantization of outputs is needed for this comparison; the interpreter's own
`input_details`/`output_details` dtype is still read and asserted against that expectation
defensively, in case a converter default ever changes.

In [17]:
N_PARITY_SAMPLES = 200
_parity_rng = np.random.default_rng(123)
_parity_idx = _parity_rng.choice(len(val_paths), size=min(N_PARITY_SAMPLES, len(val_paths)), replace=False)
_parity_paths = [val_paths[i] for i in _parity_idx]
_parity_labels = y_val[_parity_idx]

assert_val_only(_parity_paths, label="parity-check VAL sample")

_parity_images = np.stack(
    [_decode_and_preprocess(tf.constant(p)).numpy() for p in _parity_paths], axis=0
).astype(np.float32)
print(f"Parity-check sample: {_parity_images.shape}")

_keras_parity_probs = CALIBRATED_MODEL.predict(_parity_images, verbose=0, batch_size=32).astype(np.float64)
_keras_parity_argmax = _keras_parity_probs.argmax(axis=1)

def run_tflite_interpreter(tflite_path, images):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    in_detail = interpreter.get_input_details()[0]
    out_detail = interpreter.get_output_details()[0]
    in_dtype = in_detail["dtype"]
    out_dtype = out_detail["dtype"]
    in_scale, in_zero = in_detail.get("quantization", (0.0, 0))
    out_scale, out_zero = out_detail.get("quantization", (0.0, 0))

    probs_out = np.zeros((len(images), out_detail["shape"][-1]), dtype=np.float64)
    for i, img in enumerate(images):
        x = img[np.newaxis, ...]
        if np.issubdtype(in_dtype, np.integer):
            # defensive branch - not expected to trigger given the float32 I/O decision above,
            # but handled correctly (scale/zero-point) in case a converter default changes.
            x = np.round(x / in_scale + in_zero).astype(in_dtype)
        else:
            x = x.astype(in_dtype)
        interpreter.set_tensor(in_detail["index"], x)
        interpreter.invoke()
        y = interpreter.get_tensor(out_detail["index"])[0]
        if np.issubdtype(out_dtype, np.integer):
            y = (y.astype(np.float64) - out_zero) * out_scale
        else:
            y = y.astype(np.float64)
        probs_out[i] = y
    return probs_out, in_dtype, out_dtype

parity_rows = []
TFLITE_PARITY_PROBS = {}
for variant in TFLITE_VARIANTS:
    _t0 = time.time()
    probs, in_dtype, out_dtype = run_tflite_interpreter(TFLITE_PATHS[variant], _parity_images)
    TFLITE_PARITY_PROBS[variant] = probs
    abs_diff = np.abs(probs - _keras_parity_probs)
    max_abs_diff = float(abs_diff.max())
    mean_abs_diff = float(abs_diff.mean())
    argmax_agree = float((probs.argmax(axis=1) == _keras_parity_argmax).mean())
    parity_rows.append({
        "variant": variant, "input_dtype": str(in_dtype), "output_dtype": str(out_dtype),
        "max_abs_diff": max_abs_diff, "mean_abs_diff": mean_abs_diff,
        "argmax_agreement_rate": argmax_agree,
    })
    print(f"[{variant:9s}] in={str(in_dtype):8s} out={str(out_dtype):8s} "
          f"max_abs_diff={max_abs_diff:.3e} mean_abs_diff={mean_abs_diff:.3e} "
          f"argmax_agreement={argmax_agree:.4f}  ({time.time()-_t0:.1f}s)")
    if variant == "float32" and max_abs_diff > 1e-3:
        print(f"  [ESCALATION] float32 TFLite variant shows max_abs_diff={max_abs_diff:.3e} > 1e-3 "
              "vs Keras - this indicates a CONVERSION BUG, not a quantization effect (float32 "
              "should be near-bitwise-identical to the source Keras model).")

parity_df = pd.DataFrame(parity_rows)
p_parity = os.path.join(OUT_DIR, "nb07_parity_check.csv")
parity_df.to_csv(p_parity, index=False)
track_write(p_parity)
print()
print("saved:", p_parity)
print(parity_df.to_string(index=False))


Parity-check sample: (200, 96, 96, 3)


/home/yasinduslpredetor/miniconda3/envs/maternalink-fer-gpu/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


[float32  ] in=<class 'numpy.float32'> out=<class 'numpy.float32'> max_abs_diff=3.400e-03 mean_abs_diff=2.951e-04 argmax_agreement=0.9950  (0.5s)
  [ESCALATION] float32 TFLite variant shows max_abs_diff=3.400e-03 > 1e-3 vs Keras - this indicates a CONVERSION BUG, not a quantization effect (float32 should be near-bitwise-identical to the source Keras model).
[float16  ] in=<class 'numpy.float32'> out=<class 'numpy.float32'> max_abs_diff=5.384e-02 mean_abs_diff=5.345e-03 argmax_agreement=0.9900  (0.4s)
[dynint8  ] in=<class 'numpy.float32'> out=<class 'numpy.float32'> max_abs_diff=3.131e-01 mean_abs_diff=1.574e-02 argmax_agreement=0.9150  (0.4s)
[fullint8 ] in=<class 'numpy.float32'> out=<class 'numpy.float32'> max_abs_diff=8.999e-01 mean_abs_diff=8.713e-02 argmax_agreement=0.7450  (0.2s)

saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb07_parity_check.csv
 variant             input_dtype            output_dtype  max_abs_diff  mean_abs_diff  argmax_agreement

## 5. Full VALIDATION accuracy comparison (3,589 images — never TEST)

Every variant (Keras calibrated + 4 TFLite variants) is run over the **entire** VAL split
(never TEST). Accuracy, macro-F1, weighted-F1, per-class F1, and ECE (recomputed per variant
from that variant's own output probabilities) are reported, plus each variant's macro-F1 delta
vs the Keras calibrated baseline, with an explicit escalation flag for `|delta| > 0.02`.

In [18]:
from sklearn.metrics import f1_score, accuracy_score

assert_val_only(val_paths, label="section 5 full VAL accuracy comparison")

# Keras calibrated model: reuse CALIBRATED_VAL_PROBS from Part A (same model, same VAL_DS -
# no need to re-run inference).
VARIANT_VAL_PROBS = {"keras_calibrated": CALIBRATED_VAL_PROBS}

for variant in TFLITE_VARIANTS:
    print(f"Running full-VAL TFLite inference: {variant} ...")
    _t0 = time.time()
    probs, _, _ = run_tflite_interpreter(TFLITE_PATHS[variant], np.stack(
        [_decode_and_preprocess(tf.constant(p)).numpy() for p in val_paths], axis=0
    ).astype(np.float32))
    VARIANT_VAL_PROBS[variant] = probs
    print(f"  done in {time.time() - _t0:.1f}s over {len(val_paths)} images")

def macro_metrics(probs, labels, class_names):
    preds = probs.argmax(axis=1)
    acc = float(accuracy_score(labels, preds))
    macro_f1 = float(f1_score(labels, preds, average="macro", zero_division=0))
    weighted_f1 = float(f1_score(labels, preds, average="weighted", zero_division=0))
    per_class = f1_score(labels, preds, average=None, labels=list(range(len(class_names))), zero_division=0)
    ece, _ = compute_ece_and_bins(probs, (preds == labels))
    return acc, macro_f1, weighted_f1, dict(zip(class_names, [float(x) for x in per_class])), ece

comparison_rows = []
per_class_f1_rows = {}
_keras_macro_f1 = None
for variant, probs in VARIANT_VAL_PROBS.items():
    acc, macro_f1, weighted_f1, per_class_f1, ece = macro_metrics(probs, y_val, CLASS_NAMES)
    per_class_f1_rows[variant] = per_class_f1
    if variant == "keras_calibrated":
        _keras_macro_f1 = macro_f1
    comparison_rows.append({
        "variant": variant, "accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1,
        "ece": ece,
    })
    print(f"[{variant:16s}] acc={acc:.4f} macro_f1={macro_f1:.4f} weighted_f1={weighted_f1:.4f} ece={ece:.4f}")

for row in comparison_rows:
    row["macro_f1_delta_vs_keras"] = row["macro_f1"] - _keras_macro_f1
    row["escalation_flag"] = bool(abs(row["macro_f1_delta_vs_keras"]) > 0.02) if row["variant"] != "keras_calibrated" else False
    if row["escalation_flag"]:
        print(f"  [ESCALATION-WORTHY] {row['variant']}: macro_f1_delta_vs_keras="
              f"{row['macro_f1_delta_vs_keras']:+.4f} exceeds the +/-0.02 flag threshold "
              "(~3x the project's 0.0065 noise floor). Flagging only - no action taken here.")

comparison_df = pd.DataFrame(comparison_rows)
print()
print(comparison_df.to_string(index=False))


Running full-VAL TFLite inference: float32 ...


/home/yasinduslpredetor/miniconda3/envs/maternalink-fer-gpu/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  done in 21.7s over 3589 images
Running full-VAL TFLite inference: float16 ...
  done in 20.3s over 3589 images
Running full-VAL TFLite inference: dynint8 ...
  done in 18.8s over 3589 images
Running full-VAL TFLite inference: fullint8 ...
  done in 16.0s over 3589 images
[keras_calibrated] acc=0.6317 macro_f1=0.6117 weighted_f1=0.6297 ece=0.0126
[float32         ] acc=0.6319 macro_f1=0.6122 weighted_f1=0.6301 ece=0.0099
[float16         ] acc=0.6322 macro_f1=0.6090 weighted_f1=0.6305 ece=0.0135
[dynint8         ] acc=0.6283 macro_f1=0.6071 weighted_f1=0.6259 ece=0.0103
[fullint8        ] acc=0.5584 macro_f1=0.4929 weighted_f1=0.5584 ece=0.1443
  [ESCALATION-WORTHY] fullint8: macro_f1_delta_vs_keras=-0.1188 exceeds the +/-0.02 flag threshold (~3x the project's 0.0065 noise floor). Flagging only - no action taken here.

         variant  accuracy  macro_f1  weighted_f1      ece  macro_f1_delta_vs_keras  escalation_flag
keras_calibrated  0.631652  0.611693     0.629695 0.012614         

In [19]:
fig, ax = plt.subplots(figsize=(13, 6))
_variants_order = list(VARIANT_VAL_PROBS.keys())
_x = np.arange(len(CLASS_NAMES))
_width = 0.8 / len(_variants_order)
_colors = plt.cm.tab10(np.linspace(0, 1, len(_variants_order)))
for i, variant in enumerate(_variants_order):
    _vals = [per_class_f1_rows[variant][c] for c in CLASS_NAMES]
    ax.bar(_x + i * _width, _vals, width=_width, label=variant, color=_colors[i])
ax.set_xticks(_x + _width * (len(_variants_order) - 1) / 2)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1)
ax.set_title("Per-class F1 by variant \u2014 VALIDATION (Keras calibrated baseline + 4 TFLite variants)")
ax.legend(ncol=len(_variants_order), fontsize=8)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
_p = os.path.join(NB07_PLOT_DIR, "per_class_f1_by_variant.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07/per_class_f1_by_variant.png


## 6. Size and latency (development-machine indicative figures)

File size on disk and compression ratio vs the float32 `.tflite` (and vs the original `.keras`
for reference), plus indicative CPU single-image latency (20 discarded warm-up runs, then
>=200 timed runs, mean and p95 reported).

**This is a development-machine indicative figure, not a device benchmark — see notebook
08** for the actual on-device measurement.

In [20]:
SIZE_ROWS = {}
_float32_size = os.path.getsize(TFLITE_PATHS["float32"])
_keras_calibrated_size = os.path.getsize(CALIBRATED_MODEL_PATH)
_keras_original_size = os.path.getsize(MODEL_PATH)

for variant in TFLITE_VARIANTS:
    sz = os.path.getsize(TFLITE_PATHS[variant])
    SIZE_ROWS[variant] = {
        "file_size_bytes": sz,
        "compression_ratio_vs_float32_tflite": _float32_size / sz,
        "compression_ratio_vs_original_keras": _keras_original_size / sz,
    }
    print(f"[{variant:9s}] {sz:>10,} bytes  "
          f"({_float32_size / sz:.2f}x vs float32 tflite, {_keras_original_size / sz:.2f}x vs original .keras)")

print()
print(f"Reference sizes: original .keras = {_keras_original_size:,} bytes, "
      f"calibrated .keras = {_keras_calibrated_size:,} bytes")

N_LATENCY_WARMUP = 20
N_LATENCY_RUNS = 200
LATENCY_ROWS = {}
_latency_sample_img = _parity_images[0:1]  # a single representative preprocessed image

for variant in TFLITE_VARIANTS:
    interpreter = tf.lite.Interpreter(model_path=TFLITE_PATHS[variant])
    interpreter.allocate_tensors()
    in_detail = interpreter.get_input_details()[0]
    out_detail = interpreter.get_output_details()[0]
    in_dtype = in_detail["dtype"]
    x = _latency_sample_img if not np.issubdtype(in_dtype, np.integer) else _latency_sample_img.astype(in_dtype)

    for _ in range(N_LATENCY_WARMUP):
        interpreter.set_tensor(in_detail["index"], x)
        interpreter.invoke()
        _ = interpreter.get_tensor(out_detail["index"])

    _times_ms = []
    for _ in range(N_LATENCY_RUNS):
        _t0 = time.perf_counter()
        interpreter.set_tensor(in_detail["index"], x)
        interpreter.invoke()
        _ = interpreter.get_tensor(out_detail["index"])
        _times_ms.append((time.perf_counter() - _t0) * 1000.0)
    _times_ms = np.array(_times_ms)
    mean_ms = float(_times_ms.mean())
    p95_ms = float(np.percentile(_times_ms, 95))
    LATENCY_ROWS[variant] = {"mean_latency_ms": mean_ms, "p95_latency_ms": p95_ms}
    print(f"[{variant:9s}] mean={mean_ms:.3f} ms  p95={p95_ms:.3f} ms  "
          f"(dev-machine indicative, CPU interpreter, n={N_LATENCY_RUNS})")

print()
print("*** development-machine indicative figure, NOT a device benchmark - see notebook 08 ***")


[float32  ]  8,956,864 bytes  (1.00x vs float32 tflite, 2.73x vs original .keras)
[float16  ]  4,578,244 bytes  (1.96x vs float32 tflite, 5.34x vs original .keras)
[dynint8  ]  2,572,184 bytes  (3.48x vs float32 tflite, 9.51x vs original .keras)
[fullint8 ]  2,776,016 bytes  (3.23x vs float32 tflite, 8.81x vs original .keras)

Reference sizes: original .keras = 24,464,001 bytes, calibrated .keras = 24,466,049 bytes


/home/yasinduslpredetor/miniconda3/envs/maternalink-fer-gpu/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


[float32  ] mean=1.778 ms  p95=2.585 ms  (dev-machine indicative, CPU interpreter, n=200)
[float16  ] mean=2.618 ms  p95=5.304 ms  (dev-machine indicative, CPU interpreter, n=200)
[dynint8  ] mean=2.248 ms  p95=3.290 ms  (dev-machine indicative, CPU interpreter, n=200)
[fullint8 ] mean=0.929 ms  p95=1.169 ms  (dev-machine indicative, CPU interpreter, n=200)

*** development-machine indicative figure, NOT a device benchmark - see notebook 08 ***


In [21]:
fig, ax = plt.subplots(figsize=(8, 6))
for variant in TFLITE_VARIANTS:
    ax.scatter(SIZE_ROWS[variant]["file_size_bytes"] / 1024.0,
               next(r["macro_f1"] for r in comparison_rows if r["variant"] == variant),
               s=90, label=variant)
    ax.annotate(variant, (SIZE_ROWS[variant]["file_size_bytes"] / 1024.0,
                           next(r["macro_f1"] for r in comparison_rows if r["variant"] == variant)),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("file size (KB)")
ax.set_ylabel("macro-F1 (VALIDATION)")
ax.set_title("Size vs accuracy \u2014 TFLite variants (VALIDATION)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
_p = os.path.join(NB07_PLOT_DIR, "size_vs_accuracy.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)

fig, ax = plt.subplots(figsize=(9, 6))
_x = np.arange(len(TFLITE_VARIANTS))
_mean_vals = [LATENCY_ROWS[v]["mean_latency_ms"] for v in TFLITE_VARIANTS]
_p95_vals = [LATENCY_ROWS[v]["p95_latency_ms"] for v in TFLITE_VARIANTS]
ax.bar(_x - 0.2, _mean_vals, width=0.4, label="mean", color="#4C72B0")
ax.bar(_x + 0.2, _p95_vals, width=0.4, label="p95", color="#DD8452")
ax.set_xticks(_x)
ax.set_xticklabels(TFLITE_VARIANTS)
ax.set_ylabel("latency (ms)")
ax.set_title("Single-image CPU latency by variant\n"
              "development-machine indicative figure, NOT a device benchmark - see notebook 08")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
_p = os.path.join(NB07_PLOT_DIR, "latency_by_variant.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07/size_vs_accuracy.png
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07/latency_by_variant.png


## 7. Export Part B comparison artifacts to `../outputs/`

In [22]:
for row in comparison_rows:
    variant = row["variant"]
    if variant == "keras_calibrated":
        row["file_size_bytes"] = _keras_calibrated_size
        row["compression_ratio"] = 1.0
        row["mean_latency_ms"] = None
        row["p95_latency_ms"] = None
    else:
        row["file_size_bytes"] = SIZE_ROWS[variant]["file_size_bytes"]
        row["compression_ratio"] = SIZE_ROWS[variant]["compression_ratio_vs_float32_tflite"]
        row["mean_latency_ms"] = LATENCY_ROWS[variant]["mean_latency_ms"]
        row["p95_latency_ms"] = LATENCY_ROWS[variant]["p95_latency_ms"]

_column_order = ["variant", "file_size_bytes", "compression_ratio", "accuracy", "macro_f1",
                  "weighted_f1", "ece", "mean_latency_ms", "p95_latency_ms",
                  "macro_f1_delta_vs_keras", "escalation_flag"]
tflite_comparison_df = pd.DataFrame(comparison_rows)[_column_order]
p_cmp = os.path.join(OUT_DIR, "nb07_tflite_comparison.csv")
tflite_comparison_df.to_csv(p_cmp, index=False)
track_write(p_cmp)
print("saved:", p_cmp)
print(tflite_comparison_df.to_string(index=False))


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb07_tflite_comparison.csv
         variant  file_size_bytes  compression_ratio  accuracy  macro_f1  weighted_f1      ece  mean_latency_ms  p95_latency_ms  macro_f1_delta_vs_keras  escalation_flag
keras_calibrated         24466049           1.000000  0.631652  0.611693     0.629695 0.012614              NaN             NaN                 0.000000            False
         float32          8956864           1.000000  0.631931  0.612179     0.630102 0.009904         1.777711        2.584673                 0.000486            False
         float16          4578244           1.956397  0.632210  0.609022     0.630463 0.013468         2.618382        5.303809                -0.002672            False
         dynint8          2572184           3.482202  0.628309  0.607091     0.625939 0.010300         2.247834        3.289887                -0.004602            False
        fullint8          2776016           3.2

## 8. Recommendation (advisory only — not frozen, not implemented downstream)

Reasoning across the size/accuracy/latency trade-off, informed by the measured numbers above
(not hardcoded, read from `comparison_rows` / `SIZE_ROWS` / `LATENCY_ROWS` computed in this
run). This is a **recommendation**, not a decision — no code below acts on it.

- **float32**: the reference point — largest file, matches Keras almost exactly (parity check
  above), and is the safest fallback if any quantized variant shows an escalation-worthy
  macro-F1 delta.
- **float16**: typically ~2x smaller than float32 with accuracy indistinguishable from
  float32 in practice (halving mantissa precision on weights/activations rarely moves argmax
  for a softmax-headed classifier) — usually the best "free" size win.
- **dynamic-range int8**: weights-only quantization, no representative dataset needed, further
  size reduction with some risk of activation-precision loss since activations stay float at
  runtime but weights are quantized — accuracy delta should be inspected against the >0.02
  escalation threshold in the exported comparison table before trusting it.
- **full-integer int8**: smallest model, but the most quantization risk (weights AND
  activations both quantized, calibrated only against a 400-image TRAIN sample) — the
  escalation flag and parity-check numbers above are the deciding evidence, not assumption.

**RECOMMENDED (not frozen — Tech Lead decision): see the printed comparison table above —
prefer the smallest variant whose `escalation_flag` is `False` and whose parity `argmax_agreement_rate`
is acceptably high (float16 is the default expectation for that combination on a typical
FER-2013 MobileNetV2 model, but this notebook`s own measured `nb07_tflite_comparison.csv` and
`nb07_parity_check.csv` are the actual evidence a Tech Lead should read before freezing this
choice, not this paragraph's prior expectation).**

## 9. Tensor specification export — the Phase 3 exit deliverable

**7-class vs 3-class tension (stated verbatim, per project instruction):** the tensor spec
exported below is **7-class** (angry, disgust, fear, happy, neutral, sad, surprise).
`docs/system/MOOD_STATE_SPEC.md` section A4 fixes a separate **3-state** `calm/neutral/distressed`
contract for the DEPLOYED evidence vector's label space — section A4.1 explicitly states that
tensor-level details (dims, normalization, TFLite I/O spec) are a Phase-3-exit deliverable NOT
fixed by A4. This notebook produces exactly that Phase-3-exit tensor-level deliverable, and it
is 7-class. The 7-class → 3-state mapping is a **separate, not-yet-decided downstream
concern** that must reconcile with this tensor spec later — no such mapping is invented or
implemented anywhere in this notebook.

In [23]:
TENSOR_SPEC = {
    "reference_variant_for_io_contract": "float32 / float16 (both share the same float32 I/O contract)",
    "input_tensor": {
        "shape": [1, INPUT_SIZE, INPUT_SIZE, 3],
        "dtype": "float32",
        "preprocessing_contract": (
            "1) decode source JPEG as single-channel grayscale (channels=1, "
            "dct_method='INTEGER_ACCURATE' for bit-exact-with-PIL decoding); "
            "2) resize to the FER-2013 native 48x48 with bilinear interpolation, round, cast to uint8; "
            "3) replicate grayscale -> 3 channels; "
            f"4) bilinear resize to the model input size {INPUT_SIZE}x{INPUT_SIZE}; "
            "5) apply tf.keras.applications.mobilenet_v2.preprocess_input, mapping pixel values "
            "from [0, 255] to [-1, 1]."
        ),
        "int8_variant_note": (
            "dynint8 and fullint8 variants share this same float32 input dtype in this notebook "
            "(inference_input_type left as float32 by explicit choice, see Part B section 3 "
            "markdown) - they do NOT require a separate quantized input contract for the "
            "comparisons run here. A true int8 I/O contract (with its own scale/zero-point) is a "
            "valid follow-on for a device-embedded harness in notebook 08, not fixed here."
        ),
    },
    "output_tensor": {
        "shape": [1, N_CLASSES],
        "dtype": "float32",
        "class_order": CLASS_NAMES,  # derived at runtime via sorted(unique class values), never hardcoded
        "semantics": "softmax probabilities, calibrated (temperature-scaled)",
    },
    "temperature": FITTED_TEMPERATURE,
    "calibration_method": "log-softmax approximation of logits (see Part A markdown); "
                           "argmax-invariance verified for all VAL samples",
    "n_classes": N_CLASSES,
    "class_label_space_tension": (
        "This tensor spec is 7-CLASS (angry, disgust, fear, happy, neutral, sad, surprise). "
        "docs/system/MOOD_STATE_SPEC.md section A4 fixes a SEPARATE 3-state "
        "calm/neutral/distressed contract for the deployed evidence vector's label space; "
        "A4.1 explicitly defers tensor-level detail (this deliverable) to Phase 3 exit. "
        "The 7-class -> 3-state mapping is a separate, not-yet-decided downstream concern that "
        "must reconcile with this spec later. No such mapping is invented or implemented here."
    ),
    "source_model": {
        "calibrated_keras_model": os.path.basename(CALIBRATED_MODEL_PATH),
        "original_finetuned_keras_model": os.path.basename(MODEL_PATH),
        "tflite_variants": {v: os.path.basename(TFLITE_PATHS[v]) for v in TFLITE_VARIANTS},
    },
}

p_spec = os.path.join(OUT_DIR, "nb07_tensor_spec.json")
with open(p_spec, "w", encoding="utf-8") as f:
    json.dump(TENSOR_SPEC, f, indent=2)
track_write(p_spec)
print("saved:", p_spec)
print(json.dumps(TENSOR_SPEC, indent=2))


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb07_tensor_spec.json
{
  "reference_variant_for_io_contract": "float32 / float16 (both share the same float32 I/O contract)",
  "input_tensor": {
    "shape": [
      1,
      96,
      96,
      3
    ],
    "dtype": "float32",
    "preprocessing_contract": "1) decode source JPEG as single-channel grayscale (channels=1, dct_method='INTEGER_ACCURATE' for bit-exact-with-PIL decoding); 2) resize to the FER-2013 native 48x48 with bilinear interpolation, round, cast to uint8; 3) replicate grayscale -> 3 channels; 4) bilinear resize to the model input size 96x96; 5) apply tf.keras.applications.mobilenet_v2.preprocess_input, mapping pixel values from [0, 255] to [-1, 1].",
    "int8_variant_note": "dynint8 and fullint8 variants share this same float32 input dtype in this notebook (inference_input_type left as float32 by explicit choice, see Part B section 3 markdown) - they do NOT require a separate quantized input 

### Tensor spec, human-readable

| field | value |
|---|---|
| input shape | `[1, 96, 96, 3]` |
| input dtype | `float32` |
| preprocessing | grayscale decode → replicate to 3ch → bilinear resize 96x96 → `mobilenet_v2.preprocess_input` (→ `[-1, 1]`) |
| output shape | `[1, 7]` |
| output dtype | `float32` (calibrated softmax probabilities) |
| output class order | angry, disgust, fear, happy, neutral, sad, surprise (sorted, derived at runtime) |
| fitted temperature `T` | see `nb07_tensor_spec.json` → `temperature` (computed this run, not hardcoded here) |
| int8 variants | float32 I/O by explicit choice in this notebook — see `input_tensor.int8_variant_note` in the exported JSON |

**7-class vs 3-class tension:** this tensor spec is 7-class. `MOOD_STATE_SPEC.md` section A4's
3-state `calm/neutral/distressed` contract is a separate, not-yet-decided downstream mapping
that must reconcile with this spec later — not addressed in this notebook.

## 10. Run metadata — fill `RUN` and save

Final step: the `RUN` dict (redefined in Section 0) is filled from the measured values above
and written via `save_run()`, exactly as notebooks 03/04/05/06 do.

In [24]:
RUN["hyperparameters"] = {
    "input_size": INPUT_SIZE,
    "channels": 3,
    "preprocessing": "grayscale replicated to 3 channels, bilinear resize to 96x96, "
                     "mobilenet_v2.preprocess_input",
    "class_names": CLASS_NAMES,
    "n_calibration_bins": N_CAL_BINS,
    "temperature_search_bounds": list(_T_BOUNDS),
    "n_representative_samples_requested": N_REPR_SAMPLES,
    "n_parity_samples": N_PARITY_SAMPLES,
    "n_latency_warmup": N_LATENCY_WARMUP,
    "n_latency_runs": N_LATENCY_RUNS,
    "converter_settings": CONVERTER_SETTINGS,
    "test_set_touch_count": 0,
    "val_set_used_for": ["calibration_fitting", "accuracy/ece_comparison", "parity_check_sample"],
    "train_set_used_for": "int8 representative dataset only (never used for accuracy metrics)",
}

RUN["metrics"] = {
    "fitted_temperature": FITTED_TEMPERATURE,
    "ece_before_calibration_val": ECE_BEFORE,
    "ece_after_calibration_val": ECE_AFTER,
    "argmax_unchanged_confirmed": ARGMAX_UNCHANGED_CONFIRMED,
    "n_argmax_mismatches": _n_mismatch,
    "baseline_val_accuracy_uncalibrated": _baseline_val_acc,
    "tflite_comparison": {
        row["variant"]: {
            "accuracy": row["accuracy"], "macro_f1": row["macro_f1"],
            "weighted_f1": row["weighted_f1"], "ece": row["ece"],
            "macro_f1_delta_vs_keras": row["macro_f1_delta_vs_keras"],
            "escalation_flag": row["escalation_flag"],
        }
        for row in comparison_rows
    },
    "parity_check": {row["variant"]: {"max_abs_diff": row["max_abs_diff"],
                                       "mean_abs_diff": row["mean_abs_diff"],
                                       "argmax_agreement_rate": row["argmax_agreement_rate"]}
                      for row in parity_rows},
}

RUN["notes"] = (
    "Notebook 07 recalibrates the fine-tuned model (temperature scaling, fitted on VALIDATION "
    "logits reconstructed via a log-softmax approximation, argmax-invariance verified for all "
    f"{len(y_val)} VAL samples), bakes T into a saved calibrated .keras model, converts it to "
    "four TFLite variants (float32/float16/dynint8/fullint8), and compares size/accuracy/ECE/"
    "latency across all variants plus the Keras baseline on the FULL VAL split (3,589 images, "
    "TEST never touched anywhere in this notebook). Exports a 7-class tensor specification as "
    "the Phase-3-exit deliverable per MOOD_STATE_SPEC.md A4.1; the 7-class vs A4's 3-state "
    "calm/neutral/distressed contract is recorded as an explicit, unresolved downstream tension, "
    "not resolved here. Latency figures are development-machine indicative only, not a device "
    "benchmark (see notebook 08)."
)

written_total = bucket_bytes(_WRITTEN_FILES)
print(f"Total bytes written by this notebook: {written_total:,}")
for _p, _sz in _WRITTEN_FILES:
    print(f"  {_sz:>10,}  {_p}")

save_run()


Total bytes written by this notebook: 43,652,885
     151,749  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07/reliability_before_after_calibration.png
  24,466,049  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_finetuned_96_calibrated.keras
   8,956,864  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_float32.tflite
   4,578,244  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_float16.tflite
   2,572,184  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_dynint8.tflite
   2,776,016  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_fullint8.tflite
         488  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb07_parity_check.csv
      50,974  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb07/per_class_f1_by_variant.png
      55,644

'/mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/run_20260828_142821.json'